In [3]:
# 📦 Importações
import os
import json
from pathlib import Path
from PIL import Image
from tqdm import tqdm


# ✅ Configurações
root_dir = Path("/Users/jorgecunha/PycharmProjects/AI_Detection_Cars/AAU2/Test/bdd100k")
save_dir = Path("yolo_dataset")

# Categorias BDD100K
categories = [
    "pedestrian", "rider", "car", "truck", "bus", "train",
    "motorcycle", "bicycle", "traffic light", "traffic sign"
]
category_to_index = {name: i for i, name in enumerate(categories)}

# Mapeamento correto dos splits
splits_with_labels = {
    "train": root_dir / "train" / "labels.json",
    "validation": root_dir / "validation" / "labels.json",
}
splits_no_labels = {
    "test": root_dir / "test" / "data"
}

# 🚀 Processa os splits com labels
for split, label_json_path in splits_with_labels.items():
    print(f"🔄 Processando {split}...")

    image_dir = label_json_path.parent / "data"
    image_out_dir = save_dir / "images" / split
    label_out_dir = save_dir / "labels" / split
    attr_out_dir = save_dir / "attrs" / split

    image_out_dir.mkdir(parents=True, exist_ok=True)
    label_out_dir.mkdir(parents=True, exist_ok=True)
    attr_out_dir.mkdir(parents=True, exist_ok=True)

    with open(label_json_path) as f:
        annotations = json.load(f)

    for item in tqdm(annotations):
        image_name = item["name"]
        image_path = image_dir / image_name
        if not image_path.exists():
            continue

        dest_image_path = image_out_dir / image_name
        if not dest_image_path.exists():
            Image.open(image_path).save(dest_image_path)

        # ✏️ Geração do label YOLO
        label_lines = []
        for obj in item["labels"]:
            if "box2d" not in obj or obj["category"] not in category_to_index:
                continue
            box = obj["box2d"]
            class_id = category_to_index[obj["category"]]
            x_center = (box["x1"] + box["x2"]) / 2 / 1280
            y_center = (box["y1"] + box["y2"]) / 2 / 720
            width = (box["x2"] - box["x1"]) / 1280
            height = (box["y2"] - box["y1"]) / 720
            label_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

        label_path = label_out_dir / (image_name.replace(".jpg", ".txt"))
        with open(label_path, "w") as f:
            f.write("\n".join(label_lines))

        # 🧠 Salva atributos globais
        attrs = item.get("attributes", {})
        attr_data = {
            "scene": attrs.get("scene", "unspecified"),
            "weather": attrs.get("weather", "unspecified"),
            "timeofday": attrs.get("timeofday", "unspecified"),
        }
        attr_path = attr_out_dir / (image_name.replace(".jpg", ".attrs.json"))
        with open(attr_path, "w") as f:
            json.dump(attr_data, f, indent=2)

# 🧪 Copia imagens do split test
print("📥 Copiando imagens do split 'test'...")
test_image_dir = splits_no_labels["test"]
test_image_out_dir = save_dir / "images" / "test"
test_image_out_dir.mkdir(parents=True, exist_ok=True)

for file in tqdm(os.listdir(test_image_dir)):
    if file.endswith(".jpg"):
        src = test_image_dir / file
        dst = test_image_out_dir / file
        if not dst.exists():
            Image.open(src).save(dst)

print("✅ Conversão finalizada com sucesso!")


🔄 Processando train...




  0%|          | 0/69863 [00:00<?, ?it/s]

  0%|          | 222/69863 [00:02<12:29, 92.96it/s]

  1%|          | 711/69863 [00:02<03:10, 363.34it/s]

  1%|▏         | 997/69863 [00:02<02:06, 542.31it/s]

  3%|▎         | 1790/69863 [00:02<00:55, 1235.83it/s]

  4%|▎         | 2450/69863 [00:02<00:36, 1862.49it/s]

  4%|▍         | 3022/69863 [00:02<00:27, 2419.57it/s]

  5%|▌         | 3731/69863 [00:03<00:20, 3210.51it/s]

  6%|▌         | 4320/69863 [00:03<00:20, 3244.62it/s]

  7%|▋         | 4827/69863 [00:03<00:18, 3575.55it/s]

  8%|▊         | 5329/69863 [00:03<00:17, 3750.71it/s]

  8%|▊         | 5808/69863 [00:03<00:16, 3979.85it/s]

  9%|▉         | 6330/69863 [00:03<00:14, 4282.94it/s]

 10%|▉         | 6821/69863 [00:03<00:15, 4036.88it/s]

 10%|█         | 7270/69863 [00:03<00:16, 3742.02it/s]

 11%|█         | 7678/69863 [00:04<00:16, 3736.87it/s]

 12%|█▏        | 8209/69863 [00:04<00:15, 4069.11it/s]

 13%|█▎        | 8788/69863 [00:04<00:13, 4408.35it/s]

 13%|█▎   

🔄 Processando validation...




  0%|          | 0/10000 [00:00<?, ?it/s]

  0%|          | 10/10000 [00:00<01:45, 94.49it/s]

  0%|          | 20/10000 [00:00<01:51, 89.81it/s]

  0%|          | 29/10000 [00:00<01:52, 88.71it/s]

  0%|          | 39/10000 [00:00<01:48, 91.81it/s]

  0%|          | 49/10000 [00:00<01:49, 90.84it/s]

  1%|          | 59/10000 [00:00<01:51, 89.29it/s]

  1%|          | 69/10000 [00:00<01:49, 90.43it/s]

  1%|          | 79/10000 [00:00<01:48, 91.67it/s]

  1%|          | 89/10000 [00:00<01:48, 91.13it/s]

  1%|          | 99/10000 [00:01<01:49, 90.21it/s]

  1%|          | 109/10000 [00:01<01:50, 89.43it/s]

  1%|          | 118/10000 [00:01<01:52, 87.71it/s]

  1%|▏         | 128/10000 [00:01<01:51, 88.33it/s]

  1%|▏         | 137/10000 [00:01<01:51, 88.41it/s]

  1%|▏         | 147/10000 [00:01<01:49, 90.20it/s]

  2%|▏         | 157/10000 [00:01<01:48, 90.66it/s]

  2%|▏         | 167/10000 [00:01<01:48, 90.73it/s]

  2%|▏         | 177/10000 [00:01<01:48, 90.14it/s]

  2%|▏     

📥 Copiando imagens do split 'test'...


100%|██████████| 293/293 [00:03<00:00, 96.82it/s]

✅ Conversão finalizada com sucesso!
